In [1]:
!pip install transformers datasets seqeval torch

In [2]:
# STEP 1: Imports
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate


In [3]:
# STEP 2: Load Dataset
dataset = load_dataset("wikiann", "en")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
# 🔥 Use more data (better learning)
dataset["train"] = dataset["train"].select(range(20000))
dataset["validation"] = dataset["validation"].select(range(2000))

In [5]:
# STEP 3: Labels
label_list = dataset["train"].features["ner_tags"].feature.names

In [6]:
# STEP 4: Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
# STEP 5: Tokenization + Label Alignment
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True,
        max_length=128
    )

    labels = []

    for i in range(len(examples["tokens"])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        word_labels = examples["ner_tags"][i]

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(word_labels[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [8]:
# STEP 6: Model (🔥 Use BERT for better accuracy)
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [10]:
# STEP 7: Training Args (balanced speed + accuracy)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,   # 🔥 increased
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    fp16=torch.cuda.is_available()
)

In [13]:
# STEP 8: Metrics
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        curr_preds = []
        curr_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                curr_preds.append(label_list[p_val])
                curr_labels.append(label_list[l_val])

        true_predictions.append(curr_preds)
        true_labels.append(curr_labels)

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [15]:
# STEP 9: Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics
)

In [16]:
# STEP 10: Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.277828,0.240209,0.815997,0.842609,0.829090
2,0.178369,0.245850,0.837128,0.847217,0.842142
3,0.137720,0.261332,0.842289,0.855725,0.848954


TrainOutput(global_step=7500, training_loss=0.22855723876953124, metrics={'train_runtime': 709.5091, 'train_samples_per_second': 84.566, 'train_steps_per_second': 10.571, 'total_flos': 3919628528640000.0, 'train_loss': 0.22855723876953124, 'epoch': 3.0})

In [17]:
# STEP 11: Evaluate the model
print(trainer.evaluate())

{'eval_loss': 0.2613321542739868, 'eval_precision': 0.8422889043963713, 'eval_recall': 0.8557249202410493, 'eval_f1': 0.8489537541761913, 'eval_runtime': 5.3011, 'eval_samples_per_second': 377.282, 'eval_steps_per_second': 47.16, 'epoch': 3.0}


In [20]:
# STEP 12: 🔥 CLEAN INFERENCE (pipeline)
ner_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)


In [35]:
def format_output(sentence, results):
    print("Input Sentence:")
    print(sentence)
    print("\nModel Predictions:")
    print("-----------------------------------")

    for item in results:
        word = item['word']
        entity = item['entity_group']
        score = round(float(item['score']), 2)

        if entity == "PER":
            label = "Person"
        elif entity == "ORG":
            label = "Organization"
        elif entity == "LOC":
            label = "Location"
        else:
            label = entity

        print(f"{word.capitalize():<12} → {label} (Confidence: {score})")

    print("-----------------------------------")

In [37]:
sentence = "Elon_Musk founded Tesla in California"

results = ner_pipeline(sentence)

format_output(sentence, results)

Input Sentence:
Elon_Musk founded Tesla in California

Model Predictions:
-----------------------------------
Elon _ musk  → Person (Confidence: 0.89)
Tesla        → Organization (Confidence: 0.95)
California   → Location (Confidence: 0.99)
-----------------------------------
